# Picking on the Same Person - Replication

See github code: https://github.com/rishibommasani/HomogenizationNeurIPS2022

In [1]:
from functools import partial, reduce
from operator import mul, add
import pandas as pd
import numpy as np

from tqdm import tqdm
import random

## Define homogenization

In [2]:
def failure_rate(predictions, true_labels):
    assert predictions.shape == true_labels.shape, f"predictions amd true labels should have the same shape, received {predictions.shape}=={true_labels.shape}"
    return (predictions != true_labels).mean(axis=0)
    return  (predictions == 0).mean() # negative outcome
# note in their paper the failure rate 
# - is more generally the number of times a model provides a negative outcome (if not ground truth available)
# - depends on features used by a model

def observed_systemic_failure_rate(aggregated_predictions):
    # aggregated predictions = predictions by all models (row= individual, col = model)
    number_positive_predictions = aggregated_predictions.sum(axis=1)
    return (number_positive_predictions == 0).mean()

def expected_systemic_failure_rate(aggregated_predictions, true_labels):
    # aggregated predictions = predictions by all models (row= individual, col = model)
    # get column-wise accuracy
    acc_per_model = pd.DataFrame(aggregated_predictions).apply(partial(failure_rate, true_labels=true_labels), axis = 0).to_numpy()
    return reduce(mul, acc_per_model)

def homogenization(aggregated_predictions, true_labels):
    return observed_systemic_failure_rate(aggregated_predictions)/expected_systemic_failure_rate(aggregated_predictions, true_labels)


minimal example

In [3]:
# 5 models, 4 individuals
predictions = np.array([[0,1,1,0,0], [1,1,1,0,1], [0,0,1,1,0], [0,0,0,0,0]])#.sum(axis=1)
true_labels = np.array([1,1,0,1])

predictions, true_labels

(array([[0, 1, 1, 0, 0],
        [1, 1, 1, 0, 1],
        [0, 0, 1, 1, 0],
        [0, 0, 0, 0, 0]]),
 array([1, 1, 0, 1]))

In [4]:
[failure_rate(predictions[:,i], true_labels) for i in range(predictions.shape[1])]

[0.5, 0.25, 0.5, 1.0, 0.5]

In [5]:
observed_systemic_failure_rate(predictions), expected_systemic_failure_rate(predictions, true_labels), homogenization(predictions, true_labels)

(0.25, 0.03125, 8.0)

## Setup

In [6]:
# models
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import GradientBoostingClassifier

from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

In [7]:
from folktables import ACSDataSource, ACSEmployment, ACSIncomePovertyRatio, ACSHealthInsurance

In [8]:
data_source = ACSDataSource(survey_year='2018', horizon='1-Year', survey='person', root_dir = './data/folktables')
print('Loading data')
acs_data = data_source.get_data(download=False)

Loading data


In [9]:
applications_list = ['employment', 'income-poverty', 'health-insurance']
applications_data = {'employment': ACSEmployment.df_to_numpy(acs_data), 
                     'income-poverty': ACSIncomePovertyRatio.df_to_numpy(acs_data),
                     'health-insurance': ACSHealthInsurance.df_to_numpy(acs_data),
                     }
N = applications_data['employment'][0].shape[0]
print([data[0].shape for data in applications_data.values()])

[(3236107, 16), (3236107, 20), (3236107, 25)]


In [10]:
models = {
    "logistic": LogisticRegression,
    "gbm": GradientBoostingClassifier,
    "svm": SVC,
    "nn": MLPClassifier,
}

seeds = list(range(5))
data_seed = 0

for name,data in applications_data.items():
    X_train, X_test, y_train, y_test, group_train, group_test = train_test_split(
            *data, test_size=0.2, random_state=data_seed
        )
    data_splitted = {
        "X_train": X_train,
        "X_test": X_test,
        "y_train": y_train,
        "y_test": y_test,
        "group_train": group_train,
        "group_test": group_test,
    }
    applications_data[name] = data_splitted

data_scales = [25000, 10000, 7500, 5000, 2500, 1000, 750, 500, 250, 100, 50, 10]

### Fixed Partition

In [11]:
# adapted from ACS_experiemnts.py (https://github.com/rishibommasani/HomogenizationNeurIPS2022/blob/main/src/ACS_experiments.py#L133)
def get_fixed_partition(data_splitted, seed, data_scale):
	rng = random.Random(seed)
	index = rng.randint(0, data_scale - 1)
	
	# shuffle train data
	X_train, y_train, z_train = data_splitted['X_train'], data_splitted['y_train'], data_splitted['group_train']		
	y_train, z_train = np.expand_dims(y_train, axis = 1), np.expand_dims(z_train, axis = 1)
	data = np.concatenate((X_train, y_train), axis = 1)
	data = np.concatenate((data, z_train), axis = 1)
	np.random.RandomState(seed=seed).shuffle(data)
	X_train, y_train, z_train = data[:, : -2], data[:, -2], data[:, -1]

	
	# get partition size and cut out randomly located block of this size
	N = len(y_train)
	block_length = N // data_scale
	start, end = block_length * index, block_length * index + block_length
	
	# copy data, overwrite train data with partition
	data_splitted = data_splitted.copy()
	data_splitted['X_train'], data_splitted['y_train'], data_splitted['group_train'] = X_train[start : end], y_train[start : end], z_train[start : end]
		
	return data_splitted

fixed = get_fixed_partition(data_splitted, seed=seeds[0], data_scale=data_scales[0]) #size = N//data_scale

Keep the partition fixed, only change random seed of the model

In [15]:
fixed_partitions = {} #fixed_partitions[application][seed][data_scale]
model_seed = 0
model_name = 'nn'

predictions_test = {}
for ds in data_scales[:5]:
    predictions_test[ds] =  {}
    for s in seeds:
        predictions_test[ds][s] =  {}
        for name, data in applications_data.items():
            print(name)
            fixed_partition = get_fixed_partition(data, seed=0, data_scale=ds)
            model = model = make_pipeline(StandardScaler(), models[model_name](random_state=s))
            model.fit(fixed_partition['X_train'], fixed_partition['y_train'])
            predictions_test[ds][s][name] = model.predict(fixed_partition['X_test'])

employment


/Users/mgorecki/opt/miniconda3/envs/monoc-py311/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


income-poverty


/Users/mgorecki/opt/miniconda3/envs/monoc-py311/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


health-insurance


...

/Users/mgorecki/opt/miniconda3/envs/monoc-py311/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


income-poverty


/Users/mgorecki/opt/miniconda3/envs/monoc-py311/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


health-insurance


/Users/mgorecki/opt/miniconda3/envs/monoc-py311/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


In [68]:
for s in seeds:
    observed_failure = observed_systemic_failure_rate(aggregated_predictions = pd.DataFrame.from_dict(predictions_test[s]))
    expected_failure = reduce(mul, [failure_rate(preds, applications_data[name]['y_test']) for name, preds in predictions_test[s].items()])

    print(f'{observed_failure:.4f}\t{expected_failure:.4f}\t{observed_failure/expected_failure:.4f}')

0.2699	0.0188	14.3303
0.2490	0.0195	12.7850
0.2542	0.0190	13.3623
0.2563	0.0198	12.9235
0.2452	0.0201	12.1945


0.018742884407810724

In [214]:
def disjoint_partition(applications_data, applications_list, seed, data_scale):
	k = len(applications_list)
	permutation = np.random.RandomState(seed=seed).permutation(data_scale)
	
	for position, (name, entries) in enumerate(applications_data.items()):
		X_train, y_train, z_train = entries['X_train'], entries['y_train'], entries['group_train']
		y_train, z_train = np.expand_dims(y_train, axis = 1), np.expand_dims(z_train, axis = 1)
		data = np.concatenate((X_train, y_train), axis = 1)
		data = np.concatenate((data, z_train), axis = 1)
		np.random.RandomState(seed=seed).shuffle(data)
		X_train, y_train, z_train = data[:, : -2], data[:, -2], data[:, -1]

		N = len(y_train)
		block_length = N // data_scale
		index = permutation[position % data_scale] # index will be equal to position for any position < data_scale
		start, end = block_length * index, block_length * index + block_length
		entries['X_train'], entries['y_train'], entries['group_train'] = X_train[start : end], y_train[start : end], z_train[start : end]
	
	return applications_data

disjoint_partition(applications_data, ['employment'], seed=0, data_scale=data_scales[0])

0


{'employment': {'X_train': array([], shape=(0, 16), dtype=float64),
  'X_test': array([[33., 21.,  5., ...,  2.,  1.,  1.],
         [17., 15.,  5., ...,  2.,  1.,  6.],
         [50., 13.,  5., ...,  2.,  2.,  8.],
         ...,
         [48., 21.,  1., ...,  2.,  1.,  1.],
         [ 0.,  0.,  5., ...,  0.,  1.,  1.],
         [52., 16.,  1., ...,  2.,  2.,  6.]]),
  'y_train': array([], dtype=float64),
  'y_test': array([ True,  True, False, ...,  True, False, False]),
  'group_train': array([], dtype=float64),
  'group_test': array([1, 6, 8, ..., 1, 1, 6])}}

In [108]:
predictions_test = {}
for model_name in tqdm(models.keys()):
    print(model_name)
    predictions_test[model_name] = {}
    for seed in seeds:
        model = make_pipeline(StandardScaler(), models[model_name](random_state=seed))

        print(model)
        # fit on data
        model.fit(X_train, y_train)
        # get predictions on test data
        predictions_test[model_name][seed] = model.predict(X_test)

GradientBoostingClassifier()

In [95]:
for data_scale in tqdm([25000, 10000, 7500, 5000, 2500, 1000, 750, 500, 250, 100, 50, 10]):
    print(data_scale)

100%|██████████| 12/12 [00:00<00:00, 75573.05it/s]

25000
10000
7500
5000
2500
1000
750
500
250
100
50
10
